
# Silver Productos

**Fuente:** `prod_dataco.brz_dataco.raw_dataco_supply_chain` (histórico completo acumulado)

**Destino:** Tabla de **Productos**, en `prod_dataco.slv_dataco`

**Grano:** un registro por `product_card_id`

## Qué hace

1. Lee todo el histórico de Bronze acumulado, no solo el último corte.
2. Deduplica por `product_card_id` — se conserva una fila por producto, manteniendo el mismo criterio del SP original.
3. Limpia y normaliza los datos:

   * Campos de texto: `trim` + mayúsculas.
   * Identificadores numéricos: `cast` a `int`.
   * `product_price`: `decimal(12,2)`.
   * `product_status`: `int`.
4. Conserva las columnas de trazabilidad provenientes de Bronze.
5. Escribe la tabla Silver mediante `overwrite` completo.

## Trazabilidad conservada

* `fecha_inicio_corte`
* `fecha_fin_corte`
* `fecha_carga`
* `id_ejecucion`

## Resultado esperado

48 productos únicos sobre 1194 filas de Bronze para el histórico actualmente cargado.

## Pendiente / conocido

El criterio de deduplicación mantiene el comportamiento del SP original y no prioriza una versión específica del registro.

Si existen múltiples registros para un mismo `product_card_id` con valores diferentes, la fila retenida puede variar entre ejecuciones.

In [0]:
# ============================================================
# Celda 1: Parámetros + lectura de Bronze
# ============================================================

dbutils.widgets.text("bronze_catalog", "prod_dataco", "Catálogo Bronze")
dbutils.widgets.text("bronze_schema", "brz_dataco", "Esquema Bronze")
dbutils.widgets.text("bronze_table", "raw_dataco_supply_chain", "Tabla Bronze")

dbutils.widgets.text("silver_catalog", "prod_dataco", "Catálogo Silver")
dbutils.widgets.text("silver_schema", "slv_dataco", "Esquema Silver")

bronze_catalog = dbutils.widgets.get("bronze_catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
bronze_table = dbutils.widgets.get("bronze_table")

silver_catalog = dbutils.widgets.get("silver_catalog")
silver_schema = dbutils.widgets.get("silver_schema")

BRONZE_TABLE_FQN = f"{bronze_catalog}.{bronze_schema}.{bronze_table}"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{silver_catalog}.{silver_schema}"
)

df_bronze = spark.table(BRONZE_TABLE_FQN)

print(f"Total filas en Bronze: {df_bronze.count()}")
print(f"Origen: {BRONZE_TABLE_FQN}")

In [0]:
# ============================================================
# INICIO DE LOG DE EJECUCIÓN — slv_products
# ============================================================

import uuid

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control",
    key="id_ejecucion",
    debugValue=""
)

LOG_ID = str(uuid.uuid4())
NOTEBOOK = "slv_products"
CAPA = "silver"

spark.sql(f"""
INSERT INTO prod_dataco.metadata.etl_log
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")

In [0]:
# ============================================================
# Celda 2: Helpers de limpieza + deduplicación
# ------------------------------------------------------------
# clean_str  -> equivalente a UPPER(TRIM(ISNULL(col, '')))
# clean_num  -> equivalente a CAST(ISNULL(NULLIF(TRIM(col),''),'0') AS tipo)
# dedup      -> equivalente a ROW_NUMBER() OVER (PARTITION BY ... ORDER BY
#               (SELECT NULL)) = 1 del SP original
# ============================================================

from pyspark.sql import functions as F, Window

def clean_str(col_name: str, default: str = "") -> F.Column:
    return F.upper(F.trim(F.coalesce(F.col(col_name).cast("string"), F.lit(default))))

def clean_num(col_name: str, cast_type, default: str = "0") -> F.Column:
    trimmed = F.trim(F.col(col_name).cast("string"))
    nulled = F.when((trimmed.isNull()) | (trimmed == ""), None).otherwise(trimmed)
    return F.coalesce(nulled, F.lit(default)).cast(cast_type)

def dedup(df, partition_cols: list):
    w = Window.partitionBy(*partition_cols).orderBy(F.monotonically_increasing_id())
    return (
        df.withColumn("rn", F.row_number().over(w))
          .filter(F.col("rn") == 1)
          .drop("rn")
    )

print("✓ Helpers definidos: clean_str, clean_num, dedup")

In [0]:
# ============================================================
# Celda 3: Construcción de md_productos + deduplicación determinista
# ------------------------------------------------------------
# Nota: uso product_category_id (no category_id) porque es la
# columna que usaba el SP original ([Product Category Id]) —
# ambas existen en el dataset y traen el mismo valor, pero
# mantengo la que coincide con el diseño original.
#
# Deduplica por product_card_id, priorizando el corte más
# reciente (fecha_fin_corte descendente) — ya NO usa el criterio
# arbitrario del SP original.
#
# Se conservan las columnas de trazabilidad provenientes
# de Bronze para mantener la trazabilidad de extremo a extremo.
# ============================================================

df_silver_productos = df_bronze.select(
    clean_num("product_card_id", "int").alias("product_card_id"),
    clean_num("product_category_id", "int").alias("product_category_id"),
    clean_str("category_name").alias("category_name"),
    clean_num("department_id", "int").alias("department_id"),
    clean_str("department_name").alias("department_name"),
    clean_str("product_name").alias("product_name"),
    clean_str("product_description").alias("product_description"),
    clean_num("product_price", "decimal(12,2)").alias("product_price"),
    clean_num("product_status", "int").alias("product_status"),

    # --------------------------------------------------------
    # Trazabilidad
    # --------------------------------------------------------
    F.col("fecha_inicio_corte"),
    F.col("fecha_fin_corte"),
    F.col("fecha_carga"),
    F.col("id_ejecucion")
)

# ------------------------------------------------------------
# Deduplicación determinista — prioriza el corte más reciente
# ------------------------------------------------------------
from pyspark.sql import Window

def dedup_reciente(df, partition_cols: list, orden_col: str):
    w = Window.partitionBy(*partition_cols).orderBy(F.col(orden_col).desc())
    return (
        df.withColumn("rn", F.row_number().over(w))
          .filter(F.col("rn") == 1)
          .drop("rn")
    )

df_silver_productos = dedup_reciente(
    df_silver_productos,
    partition_cols=["product_card_id"],
    orden_col="fecha_fin_corte"
)

print(f"Filas en Bronze (antes de dedup)                  : {df_bronze.count()}")
print(f"Filas en Silver (después del dedup determinista)  : {df_silver_productos.count()}")

display(df_silver_productos.limit(10))

In [0]:
# ============================================================
# Celda 4: Escritura a silver_productos
# ------------------------------------------------------------
# overwrite completo (equivalente al TRUNCATE+INSERT del SP
# original), igual que silver_clientes.
# ============================================================

SILVER_PRODUCTOS_FQN = f"{silver_catalog}.{silver_schema}.md_productos"

(
    df_silver_productos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_PRODUCTOS_FQN)
)

print(f"✓ {df_silver_productos.count()} filas escritas en {SILVER_PRODUCTOS_FQN}")

In [0]:
# ============================================================
# ACTUALIZACIÓN DEL LOG — EJECUCIÓN EXITOSA
# ============================================================

spark.sql(f"""
UPDATE prod_dataco.metadata.etl_log
SET estado = 'EXITOSO', fecha_fin = current_timestamp()
WHERE log_id = '{LOG_ID}'
""")

print("✓ etl_log actualizado: EXITOSO")

In [0]:
%sql
SELECT *
FROM prod_dataco.slv_dataco.md_productos
LIMIT 5;